In [37]:
import os
import json
from dotenv import load_dotenv
from google.genai import Client


load_dotenv()  # Load environment variables from .env file

True

In [ ]:
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

google_client = Client(api_key=GOOGLE_API_KEY)

### Task 1

In [ ]:
# build a tool directory to map tool names to their corresponding functions


def calculator(operation: str, num1: float, num2: float) -> float:
    """
    A simple calculator function that performs basic arithmetic operations.

    Args:
        operation (str): The operation to perform ('add', 'subtract', 'multiply', 'divide').
        num1 (float): The first number.
        num2 (float): The second number.

    Returns:
        float: The result of the arithmetic operation.

    Raises:
        ValueError: If an unsupported operation is provided or division by zero occurs.
    """
    if operation == "add":
        return num1 + num2
    elif operation == "subtract":
        return num1 - num2
    elif operation == "multiply":
        return num1 * num2
    elif operation == "divide":
        if num2 == 0:
            raise ValueError("Cannot divide by zero.")
        return num1 / num2
    else:
        raise ValueError(f"Unsupported operation: {operation}")


def weather_info(city: str) -> str:
    """
    A function that provides weather information for a given city.

    Args:
        city (str): The name of the city.
        city name must be below list :
        - New York
        - Los Angeles
        - Chicago
        - Houston
        - Phoenix

    Returns:
        str: A string containing the weather information for the specified city.
    """

    # For demonstration purposes, we'll return a mock weather report.
    # In a real-world scenario, we would integrate with a weather API to fetch actual data.
    mock_weather_data = {
        "New York": "Sunny, 25°C",
        "Los Angeles": "Cloudy, 22°C",
        "Chicago": "Rainy, 18°C",
        "Houston": "Hot, 30°C",
        "Phoenix": "Sunny, 35°C",
    }

    return mock_weather_data.get(
        city, f"Weather information for {city} is not available."
    )


def get_stock_price(ticker: str) -> float:
    """
    A function that provides the current stock price for a given ticker symbol.

    Args:
        ticker (str): The stock ticker symbol (e.g., 'AAPL' for Apple, 'GOOGL' for Alphabet).

    Returns:
        float: The current stock price.
    """

    # For demonstration purposes, we'll return a mock stock price.
    # In a real-world scenario, we would integrate with a financial API to fetch actual stock prices.
    mock_stock_prices = {
        "AAPL": 150.25,
        "GOOGL": 2800.50,
        "AMZN": 3400.75,
        "MSFT": 299.99,
        "TSLA": 720.10,
    }

    return mock_stock_prices.get(ticker, f"Stock price for {ticker} is not available.")

In [ ]:
TOOL_DIRECTORY = {
    "calculator": calculator,
    "weather_info": weather_info,
    "get_stock_price": get_stock_price,
}


TOOL_DEFINITIONS = [
    {
        "type": "function",
        "name": "calculator",
        "description": "A simple calculator that can perform basic arithmetic operations.",
        "parameters": {
            "type": "object",
            "properties": {
                "operation": {
                    "type": "string",
                    "description": "The operation to perform ('add', 'subtract', 'multiply', 'divide').",
                },
                "num1": {"type": "number", "description": "The first number."},
                "num2": {"type": "number", "description": "The second number."},
            },
            "required": ["operation", "num1", "num2"],
        },
    },
    {
        "type": "function",
        "name": "weather_info",
        "description": "Provides weather information for a given city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "The name of the city. Supported cities: New York, Los Angeles, Chicago, Houston, Phoenix.",
                }
            },
            "required": ["city"],
        },
    },
    {
        "type": "function",
        "name": "get_stock_price",
        "description": "Provides the current stock price for a given ticker symbol.",
        "parameters": {
            "type": "object",
            "properties": {
                "ticker": {
                    "type": "string",
                    "description": "The stock ticker symbol (e.g., 'AAPL' for Apple, 'GOOGL' for Alphabet). Supported tickers: AAPL, GOOGL, AMZN, MSFT, TSLA.",
                }
            },
            "required": ["ticker"],
        },
    },
]

In [ ]:
def execute_tool(
    name: str,
    arguments: dict,
) -> dict:
    tool = TOOL_DIRECTORY.get(name)

    if tool is None:
        return {"error": f"Unknown tool: {name}"}

    try:
        return tool(**arguments)

    except Exception as exc:
        return {"error": str(exc)}


MODEL = "gemini-3.5-flash-lite"


def run_agent(user_input: str):
    print("\nUSER:")
    print(user_input)

    interaction = google_client.interactions.create(
        model=MODEL,
        input=user_input,
        tools=TOOL_DEFINITIONS,
    )

    while True:
        function_calls = [
            step for step in interaction.steps if step.type == "function_call"
        ]

        if not function_calls:
            return interaction.output_text

        function_results = []

        for function_call in function_calls:
            print("\nMODEL REQUESTED:")
            print("Tool:", function_call.name)
            print("Arguments:", function_call.arguments)

            result = execute_tool(
                name=function_call.name,
                arguments=function_call.arguments,
            )

            print("Result:", result)

            function_results.append(
                {
                    "type": "function_result",
                    "name": function_call.name,
                    "call_id": function_call.id,
                    "result": [
                        {
                            "type": "text",
                            "text": json.dumps(result),
                        }
                    ],
                }
            )

        interaction = google_client.interactions.create(
            model=MODEL,
            previous_interaction_id=interaction.id,
            input=function_results,
            tools=TOOL_DEFINITIONS,
        )

In [29]:
user_query = "I want to purchase 10 shares of AAPL so please provide me the current stock price and calculate the total cost for me."

In [36]:
answer = run_agent(user_query)
print("\nFINAL ANSWER:")
print(answer)


USER:
I want to purchase 10 shares of AAPL so please provide me the current stock price and calculate the total cost for me.

MODEL REQUESTED:
Tool: get_stock_price
Arguments: {'ticker': 'AAPL'}
Result: 150.25

MODEL REQUESTED:
Tool: calculator
Arguments: {'num2': 10, 'operation': 'multiply', 'num1': 150.25}
Result: 1502.5

FINAL ANSWER:
The current stock price for AAPL is $150.25 per share. 

To purchase 10 shares, the total cost will be $1,502.50.
